<a href="https://colab.research.google.com/github/pengin-cmd/my-colab-notebooks/blob/main/%E3%82%B3%E3%83%B3%E3%83%9A2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install xgboost catboost optuna lightgbm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 6.1 MB/s eta 0:00:00


In [ ]:
!pip install optuna

In [ ]:
pip install lifetimes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 584.2/584.2 kB 19.1 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import warnings
import optuna
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import OrdinalEncoder, MinMaxScaler
from sklearn.metrics import roc_auc_score, log_loss
from sklearn.tree import DecisionTreeClassifier, export_text
from lightgbm import LGBMClassifier, early_stopping
from xgboost import XGBClassifier
from catboost import CatBoostClassifier  # 🌟 CatBoostを追加
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')

# ==========================================
# 0. 絶対にリークしない事前処理（修正版）
# ==========================================
def preprocess_safe_row_wise(df):
    df = df.copy()
    base_date = pd.to_datetime('2015-01-01')

    if 'registration_date' in df.columns:
        df['days_since_registration'] = (base_date - pd.to_datetime(df['registration_date'])).dt.days
        df = df.drop('registration_date', axis=1)

    # 💡 改善ポイント: 極端な値を取る特徴量を np.log1p (対数変換) でマイルドにする
    if all(c in df.columns for c in ['spend_wines', 'spend_meat', 'days_since_last_purchase']):
        raw_loyalty = (df['spend_wines'] + df['spend_meat']) / (df['days_since_last_purchase'] + 1)
        df['recent_loyalty'] = np.log1p(raw_loyalty) # 対数変換で外れ値を抑え込む

    if all(c in df.columns for c in ['spend_meat', 'catalog_purchases']):
        raw_meat_ratio = df['spend_meat'] / (df['catalog_purchases'] + 1)
        df['meat_catalog_ratio'] = np.log1p(raw_meat_ratio)

    if all(c in df.columns for c in ['deals_purchases', 'store_purchases', 'web_purchases', 'catalog_purchases']):
        total_purchases = df['store_purchases'] + df['web_purchases'] + df['catalog_purchases']
        df['discount_hunter'] = df['deals_purchases'] / (total_purchases + 1)

        if 'recent_loyalty' in df.columns:
            # recent_loyalty は既にLog変換されているので、そのまま掛け合わせる
            df['campaign_affinity'] = df['recent_loyalty'] * df['discount_hunter']

    # 💡 さらなる工夫: 金額系特徴量自体も分布が歪んでいることが多いので、まとめてLog変換
    spend_cols = [c for c in df.columns if c.startswith('spend_')]
    for col in spend_cols:
        df[col + '_log'] = np.log1p(df[col])

    return df

# ==========================================
# 1. Optuna (パラメータチューニング) - ラウンド数自動取得・設定版
# ==========================================
def tune_lightgbm_with_optuna(X_raw, y, n_trials=15):
    print("\n--- [1. Optuna パラメータ最適化] ---")

    def objective(trial):
        params = {
            'random_state': 42,
            'n_estimators': 2000, # 🌟 早期終了前提なので、わざと上限を大きくしておく
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
            'max_depth': trial.suggest_int('max_depth', 3, 8),
            'num_leaves': trial.suggest_int('num_leaves', 15, 63),
            'class_weight': 'balanced',
            'verbose': -1
        }

        skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
        oof = np.zeros(len(X_raw))
        fold_best_iters = [] # 🌟 各FoldでEarly Stoppingが発動した回数を保存するリスト

        for tr_idx, va_idx in skf.split(X_raw, y):
            X_tr, y_tr = X_raw.iloc[tr_idx].copy(), y.iloc[tr_idx]
            X_va, y_va = X_raw.iloc[va_idx].copy(), y.iloc[va_idx]

            med = X_tr['annual_income'].median()
            X_tr['annual_income'] = X_tr['annual_income'].fillna(med)
            X_va['annual_income'] = X_va['annual_income'].fillna(med)

            cols = ['education_level', 'marital_status']
            enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
            X_tr[cols] = enc.fit_transform(X_tr[cols])
            X_va[cols] = enc.transform(X_va[cols])

            model = LGBMClassifier(**params)
            model.fit(
                X_tr, y_tr,
                eval_set=[(X_va, y_va)],
                callbacks=[early_stopping(stopping_rounds=30, verbose=False)]
            )
            oof[va_idx] = model.predict_proba(X_va)[:, 1]

            # 🌟 ここで「何回目で学習が止まったか」を取得！
            fold_best_iters.append(model.best_iteration_)

        # 🌟 3回のFoldの平均ストップ回数を、このTrialの情報として記録しておく
        trial.set_user_attr('optimal_iterations', int(np.mean(fold_best_iters)))

        return roc_auc_score(y, oof)

    optuna.logging.set_verbosity(optuna.logging.WARNING)
    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=n_trials)

    best_params = study.best_params

    # 🌟 優勝したパラメータ群が持っている「平均ストップ回数」を取り出す
    best_n_estimators = study.best_trial.user_attrs['optimal_iterations']
    print(f"\n💡 Optunaが導き出した理想の木の数 (n_estimators): {best_n_estimators} 回")

    # 🌟 1000回固定ではなく、自動的にベストな回数をセットする！
    best_params.update({
        'random_state': 42,
        'n_estimators': best_n_estimators,
        'class_weight': 'balanced',
        'verbose': -1
    })
    return best_params

# ==========================================
# 2. 学習フェーズ (Train & Validate)
# ==========================================
def train_models(X_raw, y, lgb_best_params, seeds=[42, 2023, 777], n_splits=5):
    print(f"\n--- [2. 学習フェーズ開始 (LGBM, XGBoost, CatBoost)] ---")
    artifacts = []
    oof_preds_ensemble = np.zeros(len(X_raw))
    ratio = float(np.sum(y == 0)) / np.sum(y == 1)

    for seed in seeds:
        print(f">> Running Seed: {seed}...")
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        oof_seed = np.zeros(len(X_raw))

        for fold, (train_idx, val_idx) in enumerate(skf.split(X_raw, y)):
            X_tr, y_tr = X_raw.iloc[train_idx].copy(), y.iloc[train_idx]
            X_va, y_va = X_raw.iloc[val_idx].copy(), y.iloc[val_idx]

            # 🌟 CatBoost用の生データ（文字列）をコピーして保持
            X_tr_cat = X_tr.copy()
            X_va_cat = X_va.copy()
            categorical_cols = ['education_level', 'marital_status']

            # CatBoostでは欠損値があるとエラーになるため、文字列に変換
            X_tr_cat[categorical_cols] = X_tr_cat[categorical_cols].astype(str)
            X_va_cat[categorical_cols] = X_va_cat[categorical_cols].astype(str)

            # 共通の数値欠損値補完
            income_median = X_tr['annual_income'].median()
            X_tr['annual_income'] = X_tr['annual_income'].fillna(income_median)
            X_va['annual_income'] = X_va['annual_income'].fillna(income_median)
            X_tr_cat['annual_income'] = X_tr_cat['annual_income'].fillna(income_median)
            X_va_cat['annual_income'] = X_va_cat['annual_income'].fillna(income_median)

            # LightGBM / XGBoost 用の数値エンコーディング
            encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
            X_tr[categorical_cols] = encoder.fit_transform(X_tr[categorical_cols])
            X_va[categorical_cols] = encoder.transform(X_va[categorical_cols])

         # 168行目付近 (train_models関数の中)
            lgb_params = lgb_best_params.copy()
            lgb_params['random_state'] = seed

            # 🌟 Optunaが算出した理想の回数を変数として取り出す
            optimal_iters = lgb_params['n_estimators']

            # ▼▼ 修正後 ▼▼
            # 🌟 LightGBMだけでなく、XGBoostとCatBoostの回数もこれに合わせて完全に足並みを揃える！
            models = {
                'LightGBM': LGBMClassifier(**lgb_params),
                'XGBoost': XGBClassifier(random_state=seed, n_estimators=optimal_iters, learning_rate=0.05, max_depth=5, scale_pos_weight=ratio, eval_metric='logloss'),
                'CatBoost': CatBoostClassifier(random_state=seed, iterations=optimal_iters, learning_rate=0.05, depth=5, auto_class_weights='Balanced', cat_features=categorical_cols, verbose=False)
            }

            fold_val_preds = np.zeros(len(X_va))
            for name, model in models.items():
                if name == 'LightGBM':
                    # 🌟 eval_set と early_stopping を外す（純粋にX_trだけで学習しきる）
                    model.fit(X_tr, y_tr)
                    fold_val_preds += model.predict_proba(X_va)[:, 1] / len(models)
                elif name == 'XGBoost':
                    # 🌟 eval_set を外す
                    model.fit(X_tr, y_tr, verbose=False)
                    fold_val_preds += model.predict_proba(X_va)[:, 1] / len(models)
                elif name == 'CatBoost':
                    # 🌟 eval_set を外す
                    model.fit(X_tr_cat, y_tr, verbose=False)
                    fold_val_preds += model.predict_proba(X_va_cat)[:, 1] / len(models)



            oof_seed[val_idx] = fold_val_preds

            artifacts.append({
                'seed': seed,
                'fold': fold,
                'preprocessors': {'income_median': income_median, 'encoder': encoder, 'cat_cols': categorical_cols},
                'models': models,
                'feature_names': list(X_tr.columns)
            })

        oof_preds_ensemble += oof_seed / len(seeds)

    print(f"✅ 学習完了: 最終 OOF AUC: {roc_auc_score(y, oof_preds_ensemble):.4f}")
    return artifacts, oof_preds_ensemble

# ==========================================
# 3. 予測フェーズ (Predict)
# ==========================================
def predict_models(X_test_raw, artifacts):
    print("\n--- [3. 予測フェーズ開始] ---")
    final_test_preds = np.zeros(len(X_test_raw))
    total_models = 0

    for artifact in artifacts:
        X_te = X_test_raw.copy()
        X_te_cat = X_test_raw.copy() # 🌟 CatBoost用コピー

        preps = artifact['preprocessors']
        models = artifact['models']

        X_te['annual_income'] = X_te['annual_income'].fillna(preps['income_median'])
        X_te_cat['annual_income'] = X_te_cat['annual_income'].fillna(preps['income_median'])

        # CatBoost用は文字列化、それ以外はエンコーディング
        X_te_cat[preps['cat_cols']] = X_te_cat[preps['cat_cols']].astype(str)
        X_te[preps['cat_cols']] = preps['encoder'].transform(X_te[preps['cat_cols']])

        for name, model in models.items():
            if name == 'CatBoost':
                final_test_preds += model.predict_proba(X_te_cat)[:, 1]
            else:
                final_test_preds += model.predict_proba(X_te)[:, 1]
            total_models += 1

    final_test_preds /= total_models
    print(f"✅ 予測完了: 計 {total_models} 個のモデルによるアンサンブル予測を実行しました。")
    return final_test_preds

# ==========================================
# 4. & 5. 可視化関数 (そのまま使用可能)
# ==========================================
def analyze_error_patterns(X_raw, y, oof_preds):
    print("\n--- [4. エラー分析 (損失原因の特定)] ---")
    error_df = X_raw.copy()
    error_df['target'] = y
    error_df['pred_class'] = (oof_preds >= 0.5).astype(int)
    error_df['is_error'] = (error_df['target'] != error_df['pred_class']).astype(int)
    print(f"▼ 全体のエラー件数 (is_error = 1): {error_df['is_error'].sum()} 件 / {len(error_df)} 件")

    X_error = X_raw.copy()
    X_error['annual_income'] = X_error['annual_income'].fillna(X_error['annual_income'].median())
    cols = ['education_level', 'marital_status']
    X_error[cols] = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1).fit_transform(X_error[cols])

    error_model = DecisionTreeClassifier(max_depth=4, random_state=42, class_weight='balanced')
    error_model.fit(X_error, error_df['is_error'])

    error_importance = pd.DataFrame({'Feature': X_error.columns, 'Error_Importance': error_model.feature_importances_ * 100}).sort_values(by='Error_Importance', ascending=False).reset_index(drop=True)
    print("\n▼ 損失原因度ランキング トップ5:")
    print(error_importance.head(5))
    return error_importance

def plot_feature_importances(artifacts):
    print("\n--- [5. 特徴量重要度の算出] ---")
    f_names = artifacts[0]['feature_names']
    importance_df = pd.DataFrame({'Feature': f_names})
    total_imps = np.zeros(len(f_names))
    total_models = 0
    scaler = MinMaxScaler()

    for art in artifacts:
        for name, model in art['models'].items():
            raw_imp = model.feature_importances_
            scaled_imp = scaler.fit_transform(raw_imp.reshape(-1, 1)).flatten()
            total_imps += scaled_imp
            total_models += 1

    importance_df['Total_Average'] = (total_imps / total_models)
    importance_df['Total_Average'] = (importance_df['Total_Average'] / importance_df['Total_Average'].sum()) * 100
    importance_df = importance_df.sort_values(by='Total_Average', ascending=False).reset_index(drop=True)

    plt.figure(figsize=(10, 8))
    sns.barplot(x='Total_Average', y='Feature', data=importance_df, palette='viridis')
    plt.title('Feature Importances (Normalized Ensemble Average)', fontsize=14)
    plt.tight_layout()
    plt.show()

# ==========================================
# メイン実行ブロック
# ==========================================
if __name__ == "__main__":
    train_data = pd.read_csv('train.csv')
    test_data = pd.read_csv('test.csv')

    train_df = preprocess_safe_row_wise(train_data)
    test_df = preprocess_safe_row_wise(test_data)

    X_raw = train_df.drop(['customer_id', 'target'], axis=1)
    y = train_df['target']
    X_test_raw = test_df.drop(['customer_id'], axis=1, errors='ignore')

    best_lgb_params = tune_lightgbm_with_optuna(X_raw, y, n_trials=10)

    artifacts, oof_predictions = train_models(X_raw, y, best_lgb_params)
    final_predictions = predict_models(X_test_raw, artifacts)

    analyze_error_patterns(X_raw, y, oof_predictions)
    plot_feature_importances(artifacts)

    submission = pd.DataFrame({'customer_id': test_data['customer_id'], 'target': final_predictions})
    submission.to_csv('final_submission_trinity.csv', index=False)
    print("\n✅ 全工程完了: 'final_submission_trinity.csv' を出力しました！")


--- [1. Optuna パラメータ最適化] ---


[W 2026-06-25 15:56:07,900] Trial 3 failed with parameters: {'learning_rate': 0.02722938361700184, 'max_depth': 4, 'num_leaves': 37} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/tmp/ipykernel_1653/1486581515.py", line 87, in objective
    model.fit(
  File "/usr/local/lib/python3.12/dist-packages/lightgbm/sklearn.py", line 1560, in fit
    super().fit(
  File "/usr/local/lib/python3.12/dist-packages/lightgbm/sklearn.py", line 1049, in fit
    self._Booster = train(
                    ^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/lightgbm/engine.py", line 322, in train
    booster.update(fobj=fobj)
  File "/usr/local/lib/python3.12/dist-packages/lightgbm/basic.py", line 4155, in update
    _LIB.LGBM_BoosterUpdateOneIter(
KeyboardInterrupt
[W 2026-06-25 15:56:07

KeyboardInterrupt: 